# Data Science & AI

**Group name:** The Turtur Squad

---

## 0. Iteration setup

**Import libraries**

**Load dataset(s)**

---

## 1. Business Understanding
### Situation description


### Business objective(s)


### Business success criteria


### Data mining goal(s)


### Data mining success criteria



---

## 2. Data Understanding

### Initial EDA

**Steps performed**
* Inspected dataset structure
* Checked missing values
* Investigated target distribution
* Explored environmental variables
* Checked class balance
* Reviewed correlations and distributions


In [1]:
# CODE CELL: Initial EDA

import pandas as pd

turtur_df = pd.read_csv(
    'data/Streptopelia turtur.csv/Streptopelia turtur.csv',
    dtype={'Streptopelia turtur': 'object'}
)
habitats_df = pd.read_csv('data/habitats_cbs_2022.csv')

turtur_df = turtur_df.rename(columns={'Streptopelia turtur': 'Streptopelia_turtur'})
target_numeric = pd.to_numeric(turtur_df['Streptopelia_turtur'], errors='coerce')
unknown_target_values = target_numeric.isna().sum()
turtur_df['Streptopelia_turtur'] = target_numeric.fillna(0)
turtur_df['presence'] = (turtur_df['Streptopelia_turtur'] > 0).astype(int)

merged_df = pd.merge(
    turtur_df,
    habitats_df,
    on=['decimalLatitude', 'decimalLongitude'],
    how='inner'
)

env_cols = ['agricultural', 'built', 'coast', 'forest', 'other', 'sand/heather', 'water', 'wetland']

print('Dataset structure')
print(f'Turtle dove observations: {len(turtur_df):,} rows, 6 original columns')
print(f'Habitat data: {len(habitats_df):,} rows, {habitats_df.shape[1]} columns')
print(f'Merged data: {len(merged_df):,} rows')

print('\nMissing values')
print('Turtle dove data:')
print(turtur_df.drop(columns=['presence']).isna().sum())
print(f'Non-numeric target values converted to 0: {unknown_target_values}')
print('\nHabitat data:')
print(habitats_df.isna().sum())

print('\nTarget distribution after merging')
class_distribution = merged_df['presence'].value_counts().sort_index()
class_summary = pd.DataFrame({
    'label': ['absence', 'presence'],
    'count': class_distribution.values,
    'percent': (class_distribution.values / class_distribution.sum() * 100).round(4)
})
print(class_summary.to_string(index=False))

print('\nPresence rate by main habitat')
presence_by_habitat = merged_df.groupby('main_habitat')['presence'].agg(['count', 'sum', 'mean'])
presence_by_habitat = presence_by_habitat.rename(columns={'count': 'observations', 'sum': 'presences', 'mean': 'presence_rate'})
presence_by_habitat['presence_rate_percent'] = (presence_by_habitat['presence_rate'] * 100).round(4)
print(presence_by_habitat.sort_values('presence_rate_percent', ascending=False)[['observations', 'presences', 'presence_rate_percent']].to_string())

print('\nEnvironmental correlations with presence')
correlations = merged_df[env_cols + ['presence']].corr(numeric_only=True)['presence'].drop('presence')
print(correlations.sort_values(key=lambda values: values.abs(), ascending=False).round(4).to_string())

print('\nEnvironmental variable skewness')
print(habitats_df[env_cols].skew().sort_values(ascending=False).round(2).to_string())


Dataset structure
Turtle dove observations: 12,558,786 rows, 6 original columns
Habitat data: 2,179 rows, 11 columns
Merged data: 12,558,756 rows

Missing values
Turtle dove data:


decimalLatitude              0
decimalLongitude             0
eventDate                    0
total_observations           5
speciesgroup_observations    0
Streptopelia_turtur          0
dtype: int64
Non-numeric target values converted to 0: 4

Habitat data:
decimalLongitude    0
decimalLatitude     0
agricultural        0
built               0
coast               0
forest              0
other               0
sand/heather        0
water               0
wetland             0
main_habitat        0
dtype: int64

Target distribution after merging
   label    count  percent
 absence 12543710  99.8802
presence    15046   0.1198

Presence rate by main habitat


              observations  presences  presence_rate_percent
main_habitat                                                
coast              1098672       2516                 0.2290
sand/heather         29220         49                 0.1677
forest              631152       1034                 0.1638
agricultural       9233520      10893                 0.1180
other                40908         39                 0.0953
water               543492        234                 0.0431
built               958416        279                 0.0291
wetland              23376          2                 0.0086

Environmental correlations with presence


coast           0.0151
built          -0.0135
wetland         0.0059
agricultural   -0.0053
water          -0.0037
forest          0.0031
sand/heather    0.0018
other          -0.0003

Environmental variable skewness
wetland         9.93
sand/heather    9.14
other           7.01
water           4.46
forest          3.78
built           3.24
coast           3.21
agricultural   -0.29


**Possible visuals**
* Missing value heatmap to show whether fields need cleaning
* Class distribution chart to show the imbalance between absence and presence
* Correlation matrix for the habitat variables and the target
* Histograms to inspect skewed environmental distributions
* Boxplots to compare habitat variables and detect outliers


In [2]:
# CODE CELL: Generate visualizations


### Slide 10 - Initial Findings

**Initial Findings**
* The merged dataset contains 12,558,756 observations linked to habitat data.
* The dataset contains many observations with no turtle dove presence: 12,543,710 absences (99.88%) and 15,046 presences (0.12%).
* The target variable is highly imbalanced, so accuracy alone can be misleading for model evaluation.
* Missing data is limited: the habitat dataset has no missing values, while the turtle dove dataset has 5 missing `total_observations` values and 4 non-numeric target values that were converted to 0.
* Some habitat categories appear more associated with sightings. The highest presence rates occur in coast (0.2290%), sand/heather (0.1677%), and forest (0.1638%) areas.
* Correlations between single habitat variables and presence are weak, with coast slightly positive and built area slightly negative.
* Several environmental variables contain skewed distributions and possible outliers, especially wetland, sand/heather, other, and water.


---

## 3. Data Preparation
*Rubric: LO 6.4C (Data Science Steps)*

**Cleaning and preprocessing**<br>


In [3]:
# CODE CELL: Data cleaning and preprocessing steps


**Adjusting dataset (optional)**<br>


In [4]:
# OPTIONAL CODE CELL: Additional preprocessing steps

---

## 4. Modeling
*Rubric: LO 6.4C (Data Science Steps)*

**Model setup**
*Describe and justify the creation of your simple benchmark model*

In [5]:
# CODE CELL: Model training and setup code
# =========================
# Benchmark Model
# =========================

import pandas as pd

# Load datasets
turtur_df = pd.read_csv('data/Streptopelia turtur.csv/Streptopelia turtur.csv', dtype={'Streptopelia turtur': 'object'})
habitats_df = pd.read_csv('data/habitats_cbs_2022.csv')
turtur_df = turtur_df.rename(columns={'Streptopelia turtur': 'Streptopelia_turtur'})
turtur_df['Streptopelia_turtur'] = pd.to_numeric(turtur_df['Streptopelia_turtur'], errors='coerce').fillna(0)

# Preview datasets
print(turtur_df.head())
print(habitats_df.head())

# Merge datasets
merged_df = pd.merge(
    turtur_df,
    habitats_df,
    on=['decimalLatitude', 'decimalLongitude'],
    how='inner'
)

# Create binary target variable
merged_df['presence'] = (
    merged_df['Streptopelia_turtur'] > 0
).astype(int)

# Check class distribution
print("\nClass Distribution:")
print(merged_df['presence'].value_counts())

# Find majority class
majority_class = merged_df['presence'].mode()[0]

print(f"\nMajority Class: {majority_class}")

# Calculate benchmark accuracy
benchmark_accuracy = (
    merged_df['presence'] == majority_class
).mean()

print(f"\nBenchmark Accuracy: {benchmark_accuracy:.2%}")

   decimalLatitude  decimalLongitude   eventDate  total_observations  \
0            50.75              5.65  2010-01-01                 0.0   
1            50.75              5.65  2010-01-02                26.0   
2            50.75              5.65  2010-01-03                 3.0   
3            50.75              5.65  2010-01-04                 0.0   
4            50.75              5.65  2010-01-05                 0.0   

   speciesgroup_observations  Streptopelia_turtur  
0                          0                  0.0  
1                         26                  0.0  
2                          3                  0.0  
3                          0                  0.0  
4                          0                  0.0  
   decimalLongitude  decimalLatitude  agricultural   built  coast  forest  \
0              5.65            50.75        1.7747  0.0568    0.0  0.0616   
1              5.70            50.75        9.9751  2.6012    0.0  0.9364   
2              5.75     


Class Distribution:
presence
0    12543710
1       15046
Name: count, dtype: int64

Majority Class: 0

Benchmark Accuracy: 99.88%


**Testing and performance**<br>


In [6]:
# CODE CELL: Model evaluation code


---

## 5. Evaluation
*Rubric: LO 6.4C (Results vs. Objectives)*
### Assessment against success criteria
The **Median Model** is the winner as it resulted in the lowest MAE. We have met our business objective by identifying **15,990 Galactic Credits** as the safest standard reference price.

### Key findings and limitations
We learned that the median is a more reliable "safe guess" for skewed data. However, this naive model is limited as it ignores specific ship features like manufacturer or age.

---

## 6 Personal Contribution
*Rubric: LO 7.3P (Equal Contribution)*

| Student name | Contribution | Personal lessons learned |
| :--- | :--- | :--- |
|  |  |  |